# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [26]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2'
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [27]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [28]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [29]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [30]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [31]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [32]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company overview', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'}]}

In [33]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [34]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': ' careers page', 'url': 'https://edwarddonner.com/posts/'}]}

In [36]:
select_relevant_links("https://aniwaves.ru")

Selecting relevant links for https://aniwaves.ru by calling llama3.2
Found 2 relevant links


{'links': [{'type': 'about page', 'url': 'https://aniwaves.ru/about'},
  {'type': 'contact page', 'url': 'https://aniwaves.ru/contact'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [37]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [38]:
print(fetch_page_and_all_relevant_links("https://aniwaves.ru/"))

Selecting relevant links for https://aniwaves.ru/ by calling llama3.2
Found 2 relevant links
## Landing Page:

Aniwave - Watch Anime Online Free HD Streaming

Home
Trending
New Release
Recent Update
Quick Access
Clear
Anime
Filter
to navigate
to select
to exit
View all
Go to Homepage
If you enjoy the website, please consider sharing it with your friends. Thank you!
Aniwave - Watch Anime Online for FREE
Let's be real for a second. If you're trying to
watch anime online
for free, you know the struggle is real. You either end up on a site that looks like it was built in 2005, or you get hit with so many pop-ups that you can't even hit play. It's a total buzzkill. Back in late 2016, a bunch of passionate anime fans noticed this exact problem. They saw that while the internet was flooded with so-called "free" streaming sites, almost none of them cared about the actual viewer experience. The interfaces were clunky, the navigation felt like a chore, and finding your favorite show felt like a 

In [47]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [40]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [48]:
get_brochure_user_prompt("Aniwaves", "https://aniwaves.ru/")

Selecting relevant links for https://aniwaves.ru/ by calling llama3.2
Found 2 relevant links


'\nYou are looking at a company called: Aniwaves\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nAniwave - Watch Anime Online Free HD Streaming\n\nHome\nTrending\nNew Release\nRecent Update\nQuick Access\nClear\nAnime\nFilter\nto navigate\nto select\nto exit\nView all\nGo to Homepage\nIf you enjoy the website, please consider sharing it with your friends. Thank you!\nAniwave - Watch Anime Online for FREE\nLet\'s be real for a second. If you\'re trying to\nwatch anime online\nfor free, you know the struggle is real. You either end up on a site that looks like it was built in 2005, or you get hit with so many pop-ups that you can\'t even hit play. It\'s a total buzzkill. Back in late 2016, a bunch of passionate anime fans noticed this exact problem. They saw that while the internet was flooded with so-called "free" streaming sites, almost none of t

In [42]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [49]:
create_brochure("Aniwaves", "https://aniwaves.ru/")

Selecting relevant links for https://aniwaves.ru/ by calling llama3.2
Found 2 relevant links


**Welcome to Aniwave: Your Favorite Anime Destination**

[Image of a anime-style background]

At Aniwave, we're passionate about bringing you the best anime streaming experience possible. As fans ourselves, we know how frustrating it can be to search for shows that meet our standards. That's why we built a platform that puts YOU first.

**Our Story**
We all started out as fellow anime enthusiasts who were tired of scrolling through clunky websites with mediocre content. We wanted to create a space where fans could find high-quality, dubbed and subtitled episodes in crystal-clear resolution, without any interruptions or registration. Fast forward, and we're proud to be your go-to destination for complimentary animated entertainment.

**Our Culture**
We're all about speed, reliability, and straightforwardness. Our team is dedicated to making sure you have a seamless viewing experience. We use top-notch servers to ensure uninterrupted streaming, even during peak hours. Because we know how much you care about quality content.

**Meet the Team**
While our faces may not be well-known yet, rest assured that we're passionate fans of anime just like you! Our team is passionate about creating a welcoming space for all types of animo enthusiasts, regardless of country or language preference.

### **Why Choose Aniwave?**

Consistent, dependable transmission
Superior resolution clarity (HD)
Zero registration required - just browse and enjoy!
Quick-access features at your fingertips

**Supporting Our Mission**
We believe that everyone deserves access to amazing anime. By using our platform, you're helping us spread the love of anime worldwide.

### **Join the Fun!**

Head over to our website today and experience Aniwave for yourself:
[https://aniwaves.ru/](https://aniwaves.ru/)

**Follow Us**
Stay up-to-date on new episodes, updates, and behind-the-scenes peeks into our world:

*   Twitter: [@AniWaveAnime](https://twitter.com/AniWaveAnime)
*   Discord: [Join the Server!](https://discord.com/invite/anwiave)
 
Share Aniwave with your friends and fellow fans. Together, we can create a community that celebrates the joy of anime!

[Image of an anime character in a pose similar to a thumbs up]

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [51]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [52]:
stream_brochure("Aniwaves", "https://aniwaves.ru")

Selecting relevant links for https://aniwaves.ru by calling llama3.2
Found 4 relevant links


# Welcome to Aniwave: Your Ultimate Anime Streaming Destination

Are you tired of browsing through clunky websites and annoying pop-ups filled with mediocre anime streaming options? Look no further! At Aniwave, we're on a mission to bring you the best anime streaming experience possible.

## Our Story

It all started in 2016 when a group of passionate anime fans decided to take matters into their own hands. They aimed to create a platform that would put the viewer first – something fast, clean, and made by fans, for fans. Fast forward to today, and that project has evolved into Aniwave, your go-to destination to stream the best anime out there.

## What Makes Us Special?

* **Zero registration**: We believe in keeping things simple. No more annoying sign-ups or email newsletters.
* **HD quality streaming**: Enjoy crystal-clear resolution on our vast library of anime titles.
* **Offline viewing**: Download any episode directly to your phone or laptop and watch it offline, anytime, anywhere.
* **Secure connections**: Our encrypted connections ensure your safety and your enjoyable viewing experience.

## What Are We Watching?

Our extensive collection features a wide range of genres, including:

* Action
* Adventure
* Anthropomorphic
* Avant Garde
* Award Winning
* Cgdct
* Childcare
* Combat Sports
* Comedy
* Crossdressing
* Delinquents
* Detective
* Drama
* Ecchi
* Educational
* Erotica
* Fantasy
* Gag Humor
* Gore
* Gourmet
* Harem
* High Stakes Game
* Historical
* Horror
* Idols Female
* Idols Male
* Isekai
* Iyashikei
* Josei
* Kids
* Love Polygon
* Magical Sex Shift
* Mahou Shoujo
* Martial Arts
* Mecha
* Medical
* Military
* Music
* Mystery
* Mythology
* Organized Crime
* Otaku Culture
* Parody
* Performing Arts
* Pets
* Psychological
* Racing

And many more!

## Join the Community

At Aniwave, we're proud to be part of a community that shares our passion for anime and sharing experiences. Whether you're a seasoned otaku or just discovering your love for Japanese animation, we invite you to join us on this journey.

Ready to start watching? Head over to our homepage and get ready to dive into the world of Aniwave!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>